In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/validation/covariance/draw_counts'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# DeepRV Dowson--Landau Draw-Count Evaluation

This notebook examines how the DeepRV-to-GP moment distance changes when it is estimated from 50, 100, 250, or 500 nested DeepRV realizations. No exact GP realizations are used.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

result_directory = GENERATED / 'soay/size_only/validation/covariance/draw_counts'  # Change This If The Notebook Starts Outside The Downloaded Results Folder
result_path = result_directory / "results" / "dowson_landau_by_draw_count.csv"
summary_path = result_directory / "results" / "dowson_landau_draw_count_summary.csv"
figure_directory = result_directory / "figures"
figure_directory.mkdir(parents=True, exist_ok=True)

if not result_path.exists():
    raise FileNotFoundError(f"Set result_directory to the downloaded run folder. Missing: {result_path}")

results = pd.read_csv(result_path)
summary = pd.read_csv(summary_path)
vital_rate_order = ["Survival", "Growth", "Reproduction", "Recruit Size"]
realization_counts = sorted(results["Realizations"].unique())
colors = dict(zip(vital_rate_order, ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]))

print(f"Loaded {len(results):,} Results")
print(f"Realization Counts: {realization_counts}")
print(f"Parameter Draws Per Vital Rate: {results['Parameter Draw'].nunique():,}")

## Numerical Summary

All distances below are divided by the target covariance trace. This makes results more comparable across vital rates with different numbers of locations and GP variances.

In [ ]:
table = summary[[
    "Vital Rate",
    "Realizations",
    "Parameter Draws",
    "Number Of Locations",
    "Median Relative D Mean Squared",
    "Median Relative D Covariance Squared",
    "Median Relative D DL Squared",
    "Relative D DL Squared 5th Percentile",
    "Relative D DL Squared 95th Percentile",
]].copy()
display(table.round(4).reset_index(drop=True))

## Distance By Number Of Realizations

Each solid line is the median over the 5,000 GP parameter draws, and each shaded region spans the 5th to 95th percentiles. The dashed line in the mean panel is the exact finite-sample reference $1/R$ for a correctly distributed zero-mean process after trace normalization. Falling curves indicate finite-sample estimation error; a persistent plateau indicates a discrepancy that is not removed by increasing the number of DeepRV realizations.

In [ ]:
metrics = {
    "Relative D Mean Squared": "Mean Component",
    "Relative D Covariance Squared": "Covariance Component",
    "Relative D DL Squared": "Total Squared Distance",
}
figure, axes = plt.subplots(1, 3, figsize=(5.906, 2.35), constrained_layout=True)

for axis, (metric, title) in zip(axes, metrics.items()):
    for vital_rate in vital_rate_order:
        current = results[results["Vital Rate"] == vital_rate]
        grouped = current.groupby("Realizations")[metric]
        median = grouped.median().reindex(realization_counts).to_numpy()
        lower = grouped.quantile(0.05).reindex(realization_counts).to_numpy()
        upper = grouped.quantile(0.95).reindex(realization_counts).to_numpy()
        axis.plot(realization_counts, median, color=colors[vital_rate], marker="o", markersize=3.5, linewidth=1.3, label=vital_rate)
        axis.fill_between(realization_counts, lower, upper, color=colors[vital_rate], alpha=0.12)
    if metric == "Relative D Mean Squared":
        axis.plot(realization_counts, 1 / np.asarray(realization_counts), color="black", linestyle="--", linewidth=1.0, label="$1/R$ Reference")
    axis.set_xscale("log")
    axis.set_xticks(realization_counts, labels=realization_counts)
    axis.minorticks_off()
    axis.set_xlabel("Number Of DeepRV Realizations", fontsize=8)
    axis.set_ylabel("Relative Squared Distance", fontsize=8)
    axis.set_title(title, fontsize=9)
    axis.tick_params(labelsize=7, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
rate_handles = handles[:-1] if labels[-1] == "$1/R$ Reference" else handles
rate_labels = labels[:-1] if labels[-1] == "$1/R$ Reference" else labels
reference_handle = handles[-1:] if labels[-1] == "$1/R$ Reference" else []
reference_label = labels[-1:] if labels[-1] == "$1/R$ Reference" else []
figure.legend(rate_handles + reference_handle, rate_labels + reference_label, loc="lower center", bbox_to_anchor=(0.5, -0.14), ncol=5, frameon=False, fontsize=7)
figure.suptitle("DeepRV Distance By Number Of Realizations", fontsize=11)
#figure.savefig(figure_directory / "dowson_landau_by_draw_count.pdf", bbox_inches="tight")
#figure.savefig(figure_directory / "dowson_landau_by_draw_count.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation

- If the relative mean component follows $1/R$, its size is consistent with ordinary Monte Carlo error in estimating a zero mean.
- If the covariance and total components continue to fall substantially from 250 to 500 realizations, the distance is still strongly affected by finite-sample covariance estimation.
- If either component settles at a positive level, that is evidence of a persistent first-two-moment mismatch between DeepRV and the target GP.
- These diagnostics compare fitted multivariate-Gaussian moments. They do not establish that the full DeepRV output distribution is Gaussian.

In [ ]:
raw_metrics = {
    "D Mean Squared": "Mean Component",
    "D Covariance Squared": "Covariance Component",
    "D DL Squared": "Total Squared Distance",
}

vital_rate_order = ["Survival", "Growth", "Reproduction", "Recruit Size"]
realization_counts = sorted(results["Realizations"].unique())
colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]

figure, axes = plt.subplots(
    4,
    3,
    figsize=(5.906, 7.2),
    sharex=True,
    constrained_layout=True,
)

for row, (vital_rate, color) in enumerate(zip(vital_rate_order, colors)):
    current = results[results["Vital Rate"] == vital_rate]

    for column, (metric, title) in enumerate(raw_metrics.items()):
        axis = axes[row, column]
        grouped = current.groupby("Realizations")[metric]

        median = grouped.median().reindex(realization_counts)
        lower = grouped.quantile(0.05).reindex(realization_counts)
        upper = grouped.quantile(0.95).reindex(realization_counts)

        axis.plot(
            realization_counts,
            median,
            color=color,
            marker="o",
            markersize=3.5,
            linewidth=1.5,
        )
        axis.fill_between(
            realization_counts,
            lower,
            upper,
            color=color,
            alpha=0.15,
        )

        if row == 0:
            axis.set_title(title, fontsize=10)

        if column == 2:
            axis.annotate(
                vital_rate,
                xy=(1.05, 0.5),
                xycoords="axes fraction",
                va="center",
                fontsize=9,
            )

        axis.set_xscale("log")
        axis.set_xticks(realization_counts, labels=realization_counts)
        axis.minorticks_off()
        axis.tick_params(labelsize=8, direction="out")
        axis.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3))
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

figure.supxlabel("Number Of DeepRV Realizations", fontsize=11)
figure.supylabel("Raw Squared Distance", fontsize=11)
figure.suptitle("Raw Dowson–Landau Distance By Number Of Realizations", fontsize=11)
plt.show()


In [ ]:
from matplotlib.ticker import PercentFormatter

plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

realizations_to_plot = 500
vital_rate_order = ["Survival", "Growth", "Reproduction", "Recruit Size"]
colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]

plot_results = results[
    results["Realizations"] == realizations_to_plot
].copy()

plot_results["Scaled Covariance Distance"] = (
    plot_results["D Covariance Squared"]
    / plot_results["Target Covariance Trace"]
)

upper_limit = plot_results["Scaled Covariance Distance"].quantile(0.99)

figure, axes = plt.subplots(
    2,
    2,
    figsize=(5.906, 4.25),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for axis, vital_rate, color in zip(
    axes.flat,
    vital_rate_order,
    colors,
):
    current = plot_results[
        (plot_results["Vital Rate"] == vital_rate)
        & (plot_results["Scaled Covariance Distance"] <= upper_limit)
    ]

    axis.scatter(
        current["Lengthscale"],
        current["Scaled Covariance Distance"],
        color=color,
        s=5,
        alpha=0.25,
        edgecolors="none",
        rasterized=True,
    )

    axis.set_title(vital_rate)
    axis.set_ylim(0, upper_limit * 1.02)

    axis.tick_params(direction="out")
    axis.margins(x=0.02)
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

figure.supxlabel(r"Posterior Lengthscale", fontsize=10)
figure.supylabel(
    r"Relative Squared Covariance Distance ",
    fontsize=10,
)

figure.savefig(
    figure_directory / "scaled_covariance_distance_by_lengthscale.pdf",
    bbox_inches="tight",
)
figure.savefig(
    figure_directory / "scaled_covariance_distance_by_lengthscale.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()